# Transport Case Study — Your Group's Scenario (Weeks 11–12)

You have been handed a **pre-built, instructor-validated transport model** of a real
**groundwater-heat-exchange (GWHE)** geothermal doublet in the Limmat Valley. Your job
is **not** to build a model — it is to **run the scenario, read the result, and judge
what it means**.

**Scenario question.** A contaminant spills upgradient of a real geothermal
**doublet** — a paired *injection* well that returns clean water to the aquifer and an
*extraction* well that abstracts it. The extraction well is your **monitoring /
compliance well**.

> ### Does the contaminant reach the monitoring well — and if so, does it exceed the regulatory threshold, and when?

*This notebook supersedes the legacy build-from-scratch transport template. It is the
deliverable for the transport phase of your continuous Weeks 9–14 group project.*

## §0 — How to use this notebook

**Modify, don't build.** The grid, its local refinement around the spill → well
corridor, the steady-state flow field, and the aquifer properties are all
**instructor-provided and validated**. You do **not** create or refine the grid (local
grid refinement is fragile and has been tuned and tested for you). You change only the
**exposed scenario knobs** in §2 and **interpret** the breakthrough in §4–§5.

**What is modelled.** The doublet runs for **flow only** (clean injection +Q,
extraction −Q). The contaminant is a *separate* zero-water source (a spill) — so the
flow field is undisturbed by the contaminant, and the question is purely: *does the
plume get captured by, or drift past, the extraction well?*

**Conservative-solute framing.** Treat your contaminant as an essentially
**conservative solute** transported by advection + dispersion. Where your group's
contaminant sorbs or decays, those reaction parameters are **pinned by the instructor**
(read-only) and reported in §2 — interpret them, do not retune them.

**Locked physics.** α_L = 10 m, α_T = 1 m, n_e = 0.20 are **locked** course-wide (see
§2). The grid resolves the **longitudinal** axis (Pe_L ≤ 2); the **transverse** axis is
intrinsically under-resolved — which constrains what you may claim (see §5).

In [ ]:
# =====================================================================
# §1 — GROUP SELECTOR + setup
# =====================================================================
import sys, os, yaml
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../../../_SUPPORT/src'))
import flopy
import geopandas as gpd
from transport_base_model import build_spill_scenario, load_doublet_base, LOCKED_PARAMS
from model_io_utils import ensure_flow_model, load_base_simulation

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# ---------------------------------------------------------------------
#  >>>  SET YOUR GROUP NUMBER HERE  (0–8)  <<<
# ---------------------------------------------------------------------
GROUP_ID = 0
# (headless-validation override — students can ignore this line)
GROUP_ID = int(os.environ.get('AGM_GROUP_ID', GROUP_ID))

# --- Toggle: the heavy build is opt-in (cached by default) ---
FORCE_RERUN = False    # True -> rebuild the scenario instead of loading the cache

# --- Load this group's scenario from the shared config ---
with open('case_config_transport.yaml') as f:
    cfg = yaml.safe_load(f)
GROUPS = {o['id']: o for o in cfg['transport_scenarios']['options']}
if GROUP_ID not in GROUPS:
    raise ValueError(f"GROUP_ID {GROUP_ID} not in config (have {sorted(GROUPS)})")
scn = GROUPS[GROUP_ID]

print(f"GROUP {GROUP_ID}: {scn['title']}")
print(f"  contaminant : {scn['contaminant']}  (concession {scn['concession']})")
print(f"  scenario    : {' '.join(scn['description'].split())[:88]}...")

## §2 — Your scenario parameters

The cell below loads everything your group needs from
`case_config_transport.yaml`: the **doublet wells** (a real AWEL concession — fixed),
the **spill** (location, strength, release schedule), the **pinned reactions**, and the
**regulatory threshold**.

- **You may modify** the *exposed knobs* — source concentration, release type/duration,
  simulation horizon — and re-run (set `FORCE_RERUN = True` in §1 after a change).
- **Do not change** the locked transport physics, the doublet coordinates, or the grid.

In [ ]:
# =====================================================================
# §2 — Your scenario knobs (EXPOSED) and the locked transport physics
# =====================================================================
d       = scn['doublet']
tr      = scn['transport']
src     = scn['source']
mon     = scn['monitoring']
sim_cfg = scn['simulation']

# --- Doublet wells: instructor-provided REAL AWEL concession (do NOT move) ---
INJ_XY = (d['injection_easting'],  d['injection_northing'])    # clean injection (+Q)
EXT_XY = (d['extraction_easting'], d['extraction_northing'])   # extraction = MONITORING well (-Q)
Q      = float(d['pumping_rate_m3_d'])

# --- Spill source: location is an E/N OFFSET (m) from the monitoring well ---
SPILL_XY = (EXT_XY[0] + float(src['location']['easting']),
            EXT_XY[1] + float(src['location']['northing']))

# --- EXPOSED scenario knobs (you may tune these and re-run) ---
C_SRC      = float(src['concentration_mg_L'])     # source concentration [mg/L]
RELEASE    = {'type': src['release_type'],        # 'pulse' | 'continuous'
              'duration_days': float(src['duration_days'])}
GEOMETRY   = src['type']                           # 'point' | 'line' | 'area'
TOTAL_TIME = float(sim_cfg['duration_days'])       # simulation horizon [d]
THRESHOLD  = float(mon['threshold_mg_L'])          # regulatory limit [mg/L]

# --- Per-group reactions: instructor-pinned, read-only ---
REACTIONS = {'Kd':     float(tr['distribution_coefficient_mL_g']),
             'rho_b':  float(tr['bulk_density_kg_m3']),
             'lambda': float(tr['first_order_decay_constant_1_per_day'])}
R = (1.0 + (REACTIONS['rho_b'] / LOCKED_PARAMS['porosity']) * (REACTIONS['Kd'] * 1e-3)
     if REACTIONS['Kd'] > 0 else 1.0)

print("EXPOSED knobs (you may modify these):")
print(f"  source concentration c_src = {C_SRC:g} mg/L")
print(f"  release                    = {RELEASE['type']} for {RELEASE['duration_days']:.0f} d ({GEOMETRY} source)")
print(f"  simulation horizon         = {TOTAL_TIME:.0f} d")
print(f"  regulatory threshold       = {THRESHOLD:g} mg/L")
print("\nLOCKED transport physics (instructor-provided; do NOT change):")
print(f"  alpha_L = {LOCKED_PARAMS['alh']} m   alpha_T = {LOCKED_PARAMS['ath1']} m   n_e = {LOCKED_PARAMS['porosity']}")
print(f"  retardation R = {R:.1f}   (Kd = {REACTIONS['Kd']} mL/g, decay lambda = {REACTIONS['lambda']} /d)")
print(f"  doublet rate Q = {Q:g} m3/d   |   grid + refinement: INSTRUCTOR-PROVIDED (you neither build nor refine)")

## §3 — Build or load the scenario  *(heavy step — opt-in)*

The first run **builds** the combined doublet + spill model (refines the spill → well
corridor, then runs the coupled steady GWF → transient GWT simulation) and **caches**
it to your group workspace. Subsequent runs **load the cache** instantly. Set
`FORCE_RERUN = True` (in §1) only when you have changed an exposed knob in §2 and want
to rebuild.

In [ ]:
# =====================================================================
# §3 — Build OR load the combined doublet + spill scenario
# =====================================================================
DATA_DIR = os.path.expanduser('~/applied_groundwater_modelling_data/limmat')
case_ws  = os.path.expanduser(cfg['output']['workspace'] + str(GROUP_ID))

def _load_coarse():
    "Load the 05f-CALIBRATED flow model + GIS (only needed to BUILD the scenario)."
    csim = load_base_simulation(str(ensure_flow_model()))   # 05f-calibrated MF6/DISV field (mean K ~375 m/d, 2,160 m³/d pumping)
    cgwf = csim.get_model('limmat_valley')
    boundary = gpd.read_file(os.path.join(DATA_DIR, 'gis', 'limmat_model_boundary.gpkg'))
    rivers = gpd.read_file(os.path.join(DATA_DIR, 'gis', 'AV_Gewasser_-OGD.gpkg'))
    rivers = rivers[rivers['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
                    & rivers.intersects(boundary.geometry.iloc[0])]
    return cgwf, boundary, rivers

have_cache = os.path.exists(os.path.join(case_ws, 'base_cache.npz'))
if FORCE_RERUN or not have_cache:
    print("Building the combined doublet + spill scenario (heavy path; minutes)...")
    cgwf, boundary_gdf, river_gdf = _load_coarse()
    db = build_spill_scenario(
        cgwf, boundary_gdf, river_gdf, INJ_XY, EXT_XY, SPILL_XY,
        case_ws=case_ws, Q=Q, c_src=C_SRC, geometry=GEOMETRY,
        release=RELEASE, reactions=REACTIONS, total_time=TOTAL_TIME)
else:
    print("Loading cached scenario (set FORCE_RERUN=True in §1 to rebuild)...")
    db = load_doublet_base(case_ws)

m = db.meta
print("\nScenario base model:")
print(f"  cells (ncpl)    = {m['ncpl']}   refine radius = {m.get('refine_radius_used', '?')} m")
print(f"  corridor Pe_L   = {m['PeL_min']:.2f} .. {m['PeL_max']:.2f}   (<= 2: longitudinal axis resolved)")
print(f"  Courant Cr peak = {m['Cr']:.2f}   (nstp = {m['nstp']}{', CAPPED' if m.get('cr_capped') else ''})")
print(f"  spill cell(s)   = {db.src_cells}   monitoring (extraction) cell = {db.receptor_cell}")

## §4 — Result: breakthrough at the monitoring well

The **left** panel is the breakthrough curve — concentration at the extraction
(monitoring) well over time — against your regulatory threshold. The **right** panel
maps the plume at the end of the simulation. Read off: peak concentration, arrival
time, and whether / when the threshold is exceeded.

In [ ]:
# =====================================================================
# §4 — Breakthrough at the monitoring well + plume map + VERDICT
# =====================================================================
times, bt = db.breakthrough           # times [d], concentration [mg/L] at the well
peak   = float(bt.max())
t_peak = float(times[int(np.argmax(bt))])
above  = np.where(bt >= THRESHOLD)[0]
t_exceed = float(times[above[0]]) if len(above) else None

# --- Verdict (peak vs threshold; arrival time) ---
if peak < 1e-9 or peak < THRESHOLD * 1e-3:
    verdict = "DOES NOT REACH the well (negligible concentration)"
elif peak >= THRESHOLD:
    verdict = f"REACHES the well ABOVE the threshold (first exceedance at t = {t_exceed:.0f} d)"
elif peak >= 0.3 * THRESHOLD:
    verdict = "REACHES the well but stays BELOW the threshold (marginal)"
else:
    verdict = "REACHES the well but stays well BELOW the threshold (dilution)"

fig, (axb, axm) = plt.subplots(1, 2, figsize=(14, 5))
axb.plot(times, bt, '-', color='#d62728', lw=2, label='C at monitoring well')
axb.axhline(THRESHOLD, color='gray', ls='--', lw=1.5, label=f'threshold = {THRESHOLD:g} mg/L')
axb.plot(t_peak, peak, 'o', color='k', ms=7, label=f'peak {peak:.3g} mg/L @ {t_peak:.0f} d')
axb.set_xlabel('Time [d]'); axb.set_ylabel('Concentration at well [mg/L]')
axb.set_title('Breakthrough at the monitoring (extraction) well')
axb.legend(fontsize=9); axb.grid(True, alpha=0.3)

mg   = db.modelgrid
conc = np.clip(db.gwt.output.concentration().get_data(totim=times[-1])[0, 0, :], 0, None)
pmv  = flopy.plot.PlotMapView(modelgrid=mg, ax=axm)
ca   = pmv.plot_array(conc, cmap='viridis')
sx, sy = db.src_xy; rx, ry = db.receptor_xy; ix, iy = INJ_XY
axm.plot(sx, sy, 'v', color='white', mec='k', ms=12, label='spill source')
axm.plot(rx, ry, '^', color='red',   mec='k', ms=12, label='monitoring well (ext)')
axm.plot(ix, iy, 's', color='cyan',  mec='k', ms=10, label='injection well')
axm.set_xlim(min(sx, rx, ix) - 200, max(sx, rx, ix) + 200)
axm.set_ylim(min(sy, ry, iy) - 200, max(sy, ry, iy) + 200)
axm.set_xlabel('Easting [m]'); axm.set_ylabel('Northing [m]')
axm.set_title(f'Plume at t = {times[-1]:.0f} d')
axm.legend(loc='upper right', fontsize=8)
fig.colorbar(ca, ax=axm, shrink=0.8, label='C [mg/L]')
fig.tight_layout(); plt.show()

print(f"VERDICT (Group {GROUP_ID} — {scn['contaminant']}):")
print(f"  peak concentration at well = {peak:.4g} mg/L   (threshold {THRESHOLD:g} mg/L)")
print(f"  -> {verdict}")

## One-change flip test

Choose **one** physically defensible scenario knob that could flip your verdict: source location, source concentration, release duration, simulation horizon, or **pumping rate**. Do **not** change the locked physics (α_L, α_T, n_e, diffusion, TVD, or grid refinement) and do **not** change the instructor-pinned reactions (R / Kd or decay).

Before you rerun, predict the direction and mechanism: for example, moving the source into the capture zone, increasing concentration or release duration, extending the horizon for a retarded plume, or changing pumping rate. For a no-reach case, a valid real lever is to **increase the pumping rate to widen the capture zone until it intercepts the plume**.

Report original vs modified peak concentration, time of peak, and verdict.


## §5 — Interpret and answer the scenario question

Use the verdict above to answer, in your report:

1. **Does the contaminant reach the monitoring well?** — capture vs lateral bypass.
   Compare the spill's lateral offset to the doublet's capture-zone half-width.
2. **Does it exceed the threshold, and when?** — peak vs threshold; first-exceedance time.
3. **Why?** — tie the outcome to the controlling process: advective capture by the
   doublet, dilution, **retardation** (sorbing groups: R in §2 delays arrival), or
   **decay** (lowers the plateau / peak).

> ### ⚠️ Defensible-threshold guidance (graded)
> Your model resolves transport **along the flow axis** (Pe_L ≤ 2) but **not across
> it** — the transverse spread is set by cell size (numerical dispersion, Pe_T ≫ 2),
> not physics. Therefore:
>
> - **Defensible — report these:** *does the plume reach the well above the limit, and
>   when* (arrival / first-exceedance); *how far downgradient along the centerline* the
>   concentration stays above threshold; the *peak concentration at the receptor*.
> - **NOT defensible — do not claim:** the *contaminated-area* map, the lateral
>   threshold-**contour width**, or any exact plume footprint — these are numerical
>   artefacts at any feasible grid resolution.
>
> Stating this limitation explicitly is part of your grade (model-quality judgment).

## §6 — Deliverable checklist

For the **transport phase** of your group report (one continuous Weeks 9–14 project),
include:

- [ ] **Scenario framed** — contaminant, source (location / type / release), the
      doublet, and the threshold question.
- [ ] **Conceptual model** — flow field + capture zone, the locked transport
      parameters (α_L, α_T, n_e), and your group's pinned reactions (R, decay) with one
      sentence on why they matter.
- [ ] **Result** — breakthrough curve at the monitoring well + plume map; the
      **verdict** (reaches? exceeds? when?).
- [ ] **Interpretation** — the controlling process (capture / bypass / retardation /
      decay / dilution).
- [ ] **Defensible-threshold judgment** (§5) — state which claims your model supports
      (longitudinal / receptor) and which it does **not** (lateral contaminated area).
- [ ] **Flip-verdict test** — one physically defensible change, your predicted
      direction + mechanism before rerun, and original vs modified peak / time /
      verdict.
- [ ] **Model files** — the cached scenario in your group workspace.

Below-threshold and no-reach outcomes are graded on judgment quality and mechanism evidence (capture-zone half-width, retardation delay, dilution), not on having a high peak.

**Rubric mapping.** This phase feeds **Conceptual Model (15%)** — the assumptions you
state and defend — and **Results & Interpretation (20%)** — the breakthrough result
*and* the limitations discussion (the defensible-threshold judgment node).

**Navigation:** ← [08t_model_application](../../transport/08t_model_application.ipynb)
(worked examples) · → your **written report** (Steps 9–10).